# Routing

Optical and high-speed RF ports have specific orientation requirements to ensure that routes avoid sharp turns, which can cause signal reflections.

We offer routing functions tailored for these requirements:

- `route_bundle`: Routes multiple connections between two port groups using a bundle/river/bus router. It also accommodates waypoints and routing steps.

The most versatile function is `route_bundle`, as it handles both single and grouped routes.

In [ ]:
import gdsfactory as gf
from amf.chp import LAYER, cells, tech, PDK
from amf.chp.tech import TECH

PDK.activate()

In [ ]:
c = gf.Component()
coupler1 = c << cells.coupler()
coupler2 = c << cells.coupler()
coupler2.move((100, 50))
c

## route_bundle

To route groups of ports avoiding waveguide collisions, use `route_bundle`.

`route_bundle` uses a river/bundle/bus router.

At the moment it works only when each group of ports have the same orientation.


In [ ]:
ys_right = [0, 40, 80, 160, 200, 220]
pitch = 30.0
N = len(ys_right)
ys_left = [(i - N / 2) * pitch for i in range(N)]
layer = LAYER.WG

right_ports = [
    gf.Port(
        f"R_{i}",
        center=(0, ys_right[i]),
        width=0.5,
        orientation=180,
        layer=gf.get_layer(layer),
    )
    for i in range(N)
]
left_ports = [
    gf.Port(
        f"L_{i}",
        center=(-300, ys_left[i]),
        width=0.5,
        orientation=0,
        layer=gf.get_layer(layer),
    )
    for i in range(N)
]

# you can also mess up the port order and it will sort them by default
left_ports.reverse()

c = gf.Component()
routes = gf.routing.route_bundle(
    c,
    left_ports,
    right_ports,
    start_straight_length=50,
    sort_ports=True,
    cross_section="strip",
)
c.add_ports(left_ports)
c.add_ports(right_ports)
c

In [ ]:
xs_top = [0, 40, 80, 160, 200, 220]
pitch = 40
N = len(xs_top)
xs_bottom = [(i - N / 2) * pitch for i in range(N)]

top_ports = [
    gf.Port(
        f"top_{i}",
        center=(xs_top[i], 0),
        width=0.5,
        orientation=270,
        layer=gf.get_layer(layer),
    )
    for i in range(N)
]

bot_ports = [
    gf.Port(
        f"bot_{i}",
        center=(xs_bottom[i], -300),
        width=0.5,
        orientation=90,
        layer=gf.get_layer(layer),
    )
    for i in range(N)
]

c = gf.Component()
routes = gf.routing.route_bundle(
    c,
    top_ports,
    bot_ports,
    separation=5.0,
    end_straight_length=100,
    cross_section="strip",
)
c

`route_bundle` can also route bundles through corners


In [ ]:
import gdsfactory as gf

In [ ]:
c = gf.Component()
c1 = c << cells.coupler()
c2 = c << cells.coupler()

c2.move((200, 100))
routes = tech.route_bundle(
    c,
    [c1.ports["o3"], c1.ports["o4"]],
    [c2.ports["o1"], c2.ports["o2"]],
    cross_section="strip",
    sort_ports=True,
)
c

In [ ]:
c = gf.Component()
c1 = c << cells.pad()
c2 = c << cells.pad()
c2.move((250, 150))
routes = tech.route_bundle_metal2(
    c,
    [c1.ports["e3"]],
    [c2.ports["e1"]],
    cross_section="metal_routing",
)
c

In [ ]:
c = gf.Component()
c1 = c << cells.pad()
c2 = c << cells.pad()
c2.move((250, 150))
routes = tech.route_bundle_metal2(
    c,
    [c1.ports["e3"]],
    [c2.ports["e1"]],
    cross_section="metal_routing",
    auto_taper=False,
)
c

**Problem**

Sometimes 90 degrees routes do not have enough space for a Manhattan route

In [ ]:
c = gf.Component()
c1 = c << gf.components.nxn(east=3, ysize=20, layer=LAYER.WG)
c2 = c << gf.components.nxn(west=3, layer=LAYER.WG)
c2.move((80, 0))
c

In [ ]:
c = gf.Component()
c1 = c << gf.components.nxn(east=3, ysize=20, layer=LAYER.WG)
c2 = c << gf.components.nxn(west=3, layer=LAYER.WG)
c2.move((200, 150))
routes = tech.route_bundle(
    c,
    list(c1.ports.filter(orientation=0)),
    list(c2.ports.filter(orientation=180)),
    on_collision=None,
    cross_section="strip",
)
c

However if the y distance is too small, the routes will overlap.

**Solution**

Add Sbend routes using `route_bundle_sbend`

In [ ]:
c = gf.Component()
c1 = c << gf.components.nxn(east=3, ysize=20, layer=LAYER.WG, wg_width=TECH.width_strip)
c2 = c << gf.components.nxn(west=3, layer=LAYER.WG, wg_width=TECH.width_strip)
c2.move((200, 100))
routes = tech.route_bundle_sbend(
    c,
    c1.ports.filter(orientation=0),
    c2.ports.filter(orientation=180),
    enforce_port_ordering=False,
)
c

## route_bundle with collisions


The route bundle alsos supports avoiding obstacles.

In [ ]:
import gdsfactory as gf

c = gf.Component()
columns = 2
ptop = c.add_ref(cells.pad(), columns=columns, column_pitch=150)
pbot = c.add_ref(cells.pad(), columns=columns, column_pitch=150)

ptop.movex(300)
ptop.movey(800)

obstacle = c << gf.c.rectangle(size=(300, 100), layer=LAYER.MT2)
obstacle.ymin = pbot.ymax - 10
obstacle.xmin = pbot.xmax - 0


routes = gf.routing.route_bundle_electrical(
    c,
    pbot.ports.filter(orientation=90),
    ptop.ports.filter(orientation=270),
    start_straight_length=100,
    separation=20,
    cross_section="metal_routing",
    bboxes=[
        obstacle.bbox().enlarge(10),
        pbot.bbox(),
        ptop.bbox(),
    ],  # obstacles to avoid need to be touching the initial or end ports
    sort_ports=True,
)

c

If the obstacle does not intersect the initial or end ports, the router does not attempt to avoid it.

## route_bundle_all_angle

You can also route using diagonal routes.

In [ ]:
import gdsfactory as gf
from amf.chp import LAYER, cells, tech

c = gf.Component()
rows = 3
straight = cells.straight
w1 = c << gf.c.array(straight, rows=rows, columns=1, row_pitch=20)
w2 = c << gf.c.array(straight, rows=rows, columns=1, row_pitch=50)
w2.rotate(-30)
w2.movex(340)
p1 = list(w1.ports.filter(orientation=0))
p2 = list(w2.ports.filter(orientation=150))
p1.reverse()
p2.reverse()

tech.route_bundle_all_angle(
    c,
    p1,
    p2,
    separation=5,
)
c

You can also use it to connect rotated components that do not have a manhattan orientation (0, 90, 180, 270)

In [ ]:
c = gf.Component()

cp = cells.coupler()
cp1 = c.add_ref_off_grid(cp)  # create a virtual instance
cp2 = c.add_ref_off_grid(cp)  # create a virtual instance
cp2.move((300, 40))
cp2.rotate(30)

routes = tech.route_bundle_all_angle(
    c,
    cp1.ports.filter(orientation=0),
    [cp2.ports["o2"], cp2.ports["o1"]],
)
c

## Dubins paths

If you’re working with PIC layouts and looking for a straightforward way to optimize waveguide paths, Dubins paths (named after Lester Eli Dubins) offer an effective solution by ensuring the shortest path with minimal bending and loss.

Using Dubins paths for waveguide routing can simplify your design process significantly. Compared to traditional interconnects, Dubins paths offer shorter, more reliable routes that avoid unnecessary bending and intersections. For PIC layouts, this translates into denser, cleaner designs with improved performance.

[See blog](https://quentinwach.com/blog/2024/02/15/dubins-paths-for-waveguide-routing.html)

In [ ]:
c = gf.Component()

# Create two straight waveguides with different orientations
wg1 = c << cells.straight(length=100, width=3.2)
wg2 = c << cells.straight(length=100, width=3.2)

# Move and rotate the second waveguide
wg2.move((300, 50))
wg2.rotate(45)

# Route between the output of wg1 and input of wg2
route = gf.routing.route_dubins(
    c,
    port1=wg1.ports["o2"],
    port2=wg2.ports["o1"],
    cross_section=tech.strip(width=3.2, radius=100),
)
c

In [ ]:
c = gf.Component()

# Create two multi-port components
comp1 = c << gf.components.nxn(
    west=0, east=10, xsize=10, ysize=100, wg_width=3.2, layer=LAYER.WG
)
comp2 = c << gf.components.nxn(
    west=0, east=10, xsize=10, ysize=100, wg_width=3.2, layer=LAYER.WG
)

# Position second component
comp2.rotate(30)
comp2.move((500, -100))

# Route between corresponding ports
for i in range(10):
    port1_name = f"o{10 - i}"  # Inverted port id for port1
    port2_name = f"o{i + 1}"  # Adjusted to match available ports
    gf.routing.route_dubins(
        c,
        port1=comp1.ports[port1_name],
        port2=comp2.ports[port2_name],
        cross_section=tech.strip(width=3.2, radius=100 + i * 10),
    )
c

## auto_taper

`route_bundle` has an `auto_taper` parameter. 

For example, in the code below if you have a width mismatch between two ports, the router will automatically add a taper to transition between the two widths, only if `auto_taper=True`, otherwise it will raise an error.


In [ ]:
c = gf.Component()
s1 = c << cells.straight()
s2 = c << cells.straight(width=2)
s2.move((40, 50))
route = tech.route_bundle(
    c,
    [s1.ports["o2"]],
    [s2.ports["o1"]],
    auto_taper=True,
)
c